In [ ]:
# Select GPU
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


# Quantum simulation libraries
from qutip import (about, basis, expect, mesolve, qeye, sigmax, sigmay, sigmaz,
                   tensor)
import qutip

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Plotting libraries
import matplotlib.pyplot as plt

# Linalg libraries
import numpy as np
import h5py as hf

In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
with hf.File("trajectory.hdf5", "r") as db:
    trajectory = db["Wanted_Positions"][:]

In [ ]:
print(trajectory.shape)

In [ ]:
plt.plot(trajectory[:500, 0], trajectory[:500, 1])
plt.xlabel("x")
plt.ylabel("y")
plt.show()

# Run Simulation

In [ ]:
# Set the system parameters
N = 5

# initial state
state_list = [basis(2, 1)] + [basis(2, 0)] * (N - 1)
psi0 = tensor(state_list)

# Energy splitting term
h = 2 * np.pi * np.ones(N)

# Interaction coefficients
Jx = 1.0 * np.pi * np.ones(N)
Jy = 1.0 * np.pi * np.ones(N)
Jz = 1.0 * np.pi * np.ones(N)

In [ ]:
def hamiltonian_h(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    h = args["h"]
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]
    
    # Hamiltonian - Energy splitting terms
    H = 0
    for i in range(N):        
        H -= driving_field[0][0] * sz_list[i]
        H -= driving_field[0][1] * sz_list[i]
        H -= driving_field[0][2] * sz_list[i]
        
        driving_field = np.delete(driving_field, 0, axis=0)


    # Interaction terms
    for n in range(N - 1):
        H += -0.5 * Jx[n] * sx_list[n] * sx_list[n + 1]
        H += -0.5 * Jy[n] * sy_list[n] * sy_list[n + 1]
        H += -0.5 * Jz[n] * sz_list[n] * sz_list[n + 1]
    
    return H
    

In [ ]:
# Setup operators for individual qubits
sx_list, sy_list, sz_list = [], [], []
for i in range(N):
    op_list = [qeye(2)] * N
    op_list[i] = sigmax()
    sx_list.append(tensor(op_list))
    op_list[i] = sigmay()
    sy_list.append(tensor(op_list))
    op_list[i] = sigmaz()
    sz_list.append(tensor(op_list))
    
train_args = {
    "h": h,
    "sx_list": sx_list,
    "sy_list": sy_list,
    "sz_list": sz_list,
    "Jx": Jx,
    "Jy": Jy,
    "Jz": Jz,
    "N": N,
    "driving": trajectory[:500]
}

test_args = {
    "h": h,
    "sx_list": sx_list,
    "sy_list": sy_list,
    "sz_list": sz_list,
    "Jx": Jx,
    "Jy": Jy,
    "Jz": Jz,
    "N": N,
    "driving": trajectory[500:1000]
}

In [ ]:
times = np.linspace(1, 500, 500)
result = mesolve(hamiltonian_h, psi0, times, [], [], train_args)
# Convert states to density matrices
train_states = [s * s.dag() for s in result.states]

result = mesolve(hamiltonian_h, psi0, times, [], [], test_args)
test_states = [s * s.dag() for s in result.states]

In [ ]:
train_signal = torch.tensor(trajectory[:500]).to(device)
test_signal = torch.tensor(trajectory[500:1000]).to(device)

In [ ]:
# Observables for state description.
state_dimension = 10000

measurements = []

for _ in range(state_dimension):
    seed = np.random.randint(641)
    measurements.append(
        qutip.rand_herm(32, 0.5, dims=[[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]])
    )

In [ ]:
# Compute obserables
train_state_descriptions = np.zeros((500, state_dimension))
test_state_descriptions = np.zeros((500, state_dimension))

for i, measurement in enumerate(measurements):
    train_state_descriptions[:, i] = np.array(
        expect(measurement, train_states)
    )
    test_state_descriptions[:, i] = np.array(
        expect(measurement, test_states)
    )
    
train_state_descriptions = torch.tensor(train_state_descriptions).to(device)
test_state_descriptions = torch.tensor(test_state_descriptions).to(device)

# Fit Model

In [ ]:
class NeuralNetwork(nn.Module):
    
    def __init__(self, state_dimension: int, output_dimension: int):
        """
        Build the network.
        
        Parameters
        ----------
        state_dimension : int
                Dimension of the state representation.
                This is the input to the layer.
        output_dimension : int
                Dimension of the output being predicted.
        """
        super().__init__()
        
        self.linear_stack = nn.Sequential(
            nn.Linear(state_dimension, 500),
            nn.ReLU(),
            nn.Linear(500, output_dimension),
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        As we are doing reservoir computing, this is 
        simply a linear layer.
        """
        return self.linear_stack(x)

In [ ]:
class ReservoirDataset(Dataset):
    """
    Custom dataset for the training.
    """
    def __init__(
        self, 
        state_data: np.ndarray, 
        function_data: np.ndarray,
        prediction_length: int
    ):
        """
        Constructor for the dataset.

        Parameters
        ----------
        state_data : np.ndarray
                State description data.
        function_data : np.ndarray
                Function data being fit.
                This will be the target data.
        prediction_length : int
                How far into the future you will predict.
        """
        self.state_data = state_data
        self.function_data = function_data
        self.prediction_length = prediction_length
        
        self.norm_factor = max(abs(function_data.flatten()))
    
    def __len__(self):
        """
        Length of the dataset.
        """
        return int(
            len(self.function_data) - self.prediction_length
        )
    
    def __getitem__(self, idx: int):
        """
        Collect an item from the dataset.
        
        Parameters
        ----------
        idx : int
                Index of the state to take.
        """
        state = self.state_data[idx]
        target = self.function_data[
            int(idx + self.prediction_length)
        ] / self.norm_factor
        
        return state, target

In [ ]:
# Compile the neural network.
model = NeuralNetwork(
    state_dimension=state_dimension, 
    output_dimension=3
).to(device)
model = model.type(torch.float64)

# Use MSE loss
loss_fn = nn.MSELoss()

# Use ADAM optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

# Short model summary for sanity check.
print(model)

In [ ]:
training_data = ReservoirDataset(
    state_data=train_state_descriptions,
    function_data=train_signal,
    prediction_length=5
)
test_data = ReservoirDataset(
    state_data=test_state_descriptions,
    function_data=test_signal,
    prediction_length=5
)

In [ ]:
training_loader = DataLoader(training_data, batch_size=10, shuffle=False)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
            
    return loss

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: Avg loss: {test_loss:>8f} \n")
    
    return test_loss

In [ ]:
epochs = 400
train_losses = []
test_losses = []

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    loss = train(training_loader, model, loss_fn, optimizer)
    train_losses.append(loss)
    loss = test(test_loader, model, loss_fn)
    test_losses.append(loss)
print("Done!")
train_losses = [item.cpu().detach().numpy() for item in train_losses]

In [ ]:
plt.plot(test_losses)
plt.yscale("log")

In [ ]:
predictions = model(test_state_descriptions).cpu().detach().numpy()

In [ ]:
plt.plot(times + 5.0, predictions[:, 0] * 40.1303)
plt.plot(times, test_signal.cpu()[:, 0])
plt.show()

In [ ]:
plt.plot(times + 5.0, predictions[:, 1] * 40.1303)
plt.plot(times, test_signal.cpu()[:, 1])
plt.show()

In [ ]:
plt.plot(times + 5.0, predictions[:, 2] * 40.1303)
plt.plot(times + 5, test_signal.cpu()[:, 3])
plt.show()